# Agente RAG - Mercado Central 24h

Challenge Alura - ONE IA for Tech

La idea es armar un agente que responda preguntas sobre los documentos internos
de la empresa, para no tener que abrir los PDFs cada vez que alguien necesita algo.

En este notebook voy probando las etapas del challenge una por una. Cuando todo
funciona, paso el codigo a los archivos `crear_base.py` y `app.py` para poder
subirlo al servidor.

**Los documentos que uso:**

| Archivo | De que trata |
|---|---|
| Politica_Atencion_Cliente_Devoluciones...pdf | cambios, devoluciones, reembolsos |
| FAQ_Mercado_Central_24h.pdf | preguntas de clientes y de empleados |
| Reglamento_Interno...pdf | normas internas, horarios, permisos |
| Manual_Proveedores...pdf | compras y alta de proveedores |
| inventario_de_supermercado_latam.xlsx | 200 productos con stock y precios |

## Instalar las librerias

Esto solo hace falta la primera vez.

In [ ]:
# Si ya las tengo instaladas puedo saltarme esta celda
!pip install -q -r requirements.txt

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# La API key la tengo en el archivo .env para no dejarla escrita en el codigo
load_dotenv()

print("API key cargada:", "GOOGLE_API_KEY" in os.environ)

## Etapa 1 - Colecta y organizacion de documentos

Primero veo que archivos tengo en la carpeta `data`.

Ademas les pongo una **categoria** a cada uno (RH, atencion al cliente, etc).
Esa categoria la guardo como metadato, asi despues puedo saber de que area salio
cada respuesta.

In [ ]:
CARPETA = "data"

for archivo in sorted(os.listdir(CARPETA)):
    peso = os.path.getsize(os.path.join(CARPETA, archivo)) / 1024
    print(f"{archivo}  ({peso:.0f} KB)")

In [ ]:
CATEGORIAS = {
    "Politica_Atencion_Cliente_Devoluciones_Mercado_Central_24h.pdf": "Atencion al Cliente",
    "FAQ_Mercado_Central_24h.pdf": "Preguntas Frecuentes",
    "Reglamento_Interno_Mercado_Central_24h.pdf": "Recursos Humanos",
    "Manual_Proveedores_Mercado_Central_24h.pdf": "Compras y Proveedores",
    "inventario_de_supermercado_latam.xlsx": "Inventario",
}

CATEGORIAS

## Etapa 2 - Proceso y extraccion de contenido

Aca saco el texto de adentro de los archivos.

Los PDF y el Excel se leen distinto:

- **PDF:** uso PyPDF. Como los PDF son digitales (no escaneados) el texto sale
  directo, no necesito OCR. Guardo tambien el numero de pagina para poder citar
  la fuente despues.
- **Excel:** una tabla no es texto corrido, asi que convierto cada fila en una
  frase repitiendo el nombre de cada columna. Por ejemplo:
  `SKU: MER-003, Descripcion: Arroz Integral 1kg, Stock Actual: 45, ...`

In [ ]:
def leer_pdfs():
    documentos = []

    for archivo in sorted(os.listdir(CARPETA)):
        if not archivo.endswith(".pdf"):
            continue

        loader = PyPDFLoader(os.path.join(CARPETA, archivo))
        paginas = loader.load()

        for pagina in paginas:
            pagina.metadata["archivo"] = archivo
            pagina.metadata["categoria"] = CATEGORIAS[archivo]
            # PyPDF empieza a contar en 0, le sumo 1
            pagina.metadata["pagina"] = pagina.metadata["page"] + 1

        print(archivo, "->", len(paginas), "paginas")
        documentos += paginas

    return documentos


docs_pdf = leer_pdfs()
print("\nTotal de paginas:", len(docs_pdf))

In [ ]:
# Miro como quedo una pagina cualquiera
print(docs_pdf[5].metadata)
print()
print(docs_pdf[5].page_content[:400])

In [ ]:
def leer_inventario():
    archivo = "inventario_de_supermercado_latam.xlsx"
    df = pd.read_excel(os.path.join(CARPETA, archivo))

    documentos = []
    for _, fila in df.iterrows():
        texto = ", ".join(f"{col}: {fila[col]}" for col in df.columns)
        documentos.append(
            Document(
                page_content="Producto del inventario. " + texto,
                metadata={
                    "archivo": archivo,
                    "categoria": CATEGORIAS[archivo],
                    "pagina": 0,
                },
            )
        )
    return documentos


docs_excel = leer_inventario()
print("Productos:", len(docs_excel))
print()
print(docs_excel[2].page_content[:300])

In [ ]:
documentos = docs_pdf + docs_excel
print("Documentos en total:", len(documentos))

### Cortar el texto en chunks

No puedo mandarle un PDF entero al modelo, no le entra y ademas se confunde.
Entonces corto todo en pedacitos de mas o menos 1000 caracteres.

Le pongo un `chunk_overlap` de 150 para que los pedazos se pisen un poco entre si.
Si corto justo en la mitad de una frase, con el solapamiento no pierdo el sentido.

In [ ]:
divisor = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = divisor.split_documents(documentos)

print("Chunks:", len(chunks))
print()
print("Ejemplo de chunk:")
print(chunks[10].page_content[:300])
print()
print("Metadatos:", chunks[10].metadata)

## Etapa 3 - Indexacion

Ahora convierto cada chunk en un **embedding**, que es una lista de numeros que
representa el significado del texto.

Lo bueno de esto es que dos textos que quieren decir lo mismo quedan con numeros
parecidos, aunque usen palabras distintas. Por eso despues puedo preguntar
"cuantos dias de vacaciones tengo" y encuentra un parrafo que habla de
"licencia remunerada".

Los guardo en **FAISS**, que es una base de datos vectorial que corre local y es
gratis.

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# Esto tarda un par de minutos porque son muchos chunks
base = FAISS.from_documents(chunks, embeddings)
base.save_local("vectorstore")

print("Base creada y guardada en la carpeta vectorstore/")

In [ ]:
# Para no tener que crear la base de nuevo cada vez que abro el notebook,
# la puedo cargar desde el disco
base = FAISS.load_local("vectorstore", embeddings, allow_dangerous_deserialization=True)
print("Base cargada")

## Etapa 4 - Camada de recuperacion (RAG)

Aca pruebo si la busqueda funciona bien, todavia sin usar el modelo de lenguaje.

Le paso una pregunta y me tiene que devolver los pedazos de documento que hablan
del tema. Le pido 4 resultados (`k=4`).

In [ ]:
retriever = base.as_retriever(search_kwargs={"k": 4})

resultados = retriever.invoke("Cual es la politica de devoluciones?")

for r in resultados:
    print("Archivo:", r.metadata["archivo"], "| pagina:", r.metadata["pagina"])
    print(r.page_content[:200].replace("\n", " "))
    print("-" * 70)

In [ ]:
# Pruebo con el inventario, que es la parte que viene del Excel
resultados = retriever.invoke("Cuantas unidades de arroz integral hay?")

for r in resultados:
    print(r.metadata["archivo"], "|", r.page_content[:180])
    print("-" * 70)

## Etapa 5 - Produccion y validacion de respuestas

Ahora si armo la respuesta final.

En el prompt le aclaro dos cosas importantes al modelo:

1. Que use **solo** el contexto que le paso, nada de conocimiento propio.
2. Que si no encuentra la respuesta, lo diga en vez de inventar.

Esto es para evitar las alucinaciones. Ademas devuelvo siempre el archivo y la
pagina de donde salio, asi cualquiera puede ir a verificarlo.

In [ ]:
modelo = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)

PROMPT = """Eres el asistente virtual de Mercado Central 24h.
Responde la pregunta usando solamente la informacion del contexto de abajo.
Si la respuesta no aparece en el contexto, responde exactamente:
"No encontre esa informacion en los documentos de Mercado Central 24h."
No inventes datos. Responde en espanol, claro y breve.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""

In [ ]:
def responder(pregunta):
    # 1. buscar los fragmentos relevantes
    encontrados = retriever.invoke(pregunta)
    contexto = "\n\n".join(d.page_content for d in encontrados)

    # 2. pedirle la respuesta al modelo
    salida = modelo.invoke(PROMPT.format(contexto=contexto, pregunta=pregunta))

    # 3. armar la lista de fuentes sin repetir
    fuentes = []
    for d in encontrados:
        f = d.metadata["archivo"]
        if d.metadata["pagina"]:
            f += f" (pag. {d.metadata['pagina']})"
        if f not in fuentes:
            fuentes.append(f)

    return salida.content, fuentes

In [ ]:
respuesta, fuentes = responder("Cual es la politica de devoluciones?")

print(respuesta)
print()
print("Fuentes:", fuentes)

In [ ]:
respuesta, fuentes = responder("Que beneficios tiene el programa Cliente VIP Central?")

print(respuesta)
print()
print("Fuentes:", fuentes)

In [ ]:
respuesta, fuentes = responder("Cuantas unidades de Arroz Integral 1kg hay en el inventario?")

print(respuesta)
print()
print("Fuentes:", fuentes)

### Probar que no invente cosas

Le pregunto algo que seguro no esta en ningun documento, para ver si contesta
que no lo encontro en vez de inventar.

In [ ]:
respuesta, fuentes = responder("Cuantos empleados tiene la sucursal de Tokio?")
print(respuesta)

## Conclusiones y siguientes pasos

El agente ya funciona: encuentra la informacion en los documentos, responde en
lenguaje natural y muestra de donde saco cada dato. Tambien probe que cuando la
respuesta no esta, avisa en vez de inventarla.

Lo que sigue:

1. Pasar este codigo a `crear_base.py` y `app.py` (ya esta hecho).
2. Levantar la interfaz con `streamlit run app.py` para probarla como pagina web.
3. Etapa 7: subir todo a una maquina de Oracle Cloud para que quede publico.

**Nota:** la app guarda cada pregunta en `registro_preguntas.jsonl` (etapa 8),
con la fecha, la respuesta, las fuentes y cuanto tardo. Eso me sirve para
revisar despues si el agente esta respondiendo bien.